In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Create Widgets ---
dbutils.widgets.removeAll()
dbutils.widgets.text("scale", "1000", "Scale Factor (GB)")
dbutils.widgets.text("dbx_warehouse_url", "", "Databricks Warehouse JDBC URL")
dbutils.widgets.text("dbx_pat_token", "", "Databricks PAT Token")
dbutils.widgets.text("dbx_optimize_strategy", "optimize", "Optimize Strategy")
dbutils.widgets.text("dbx_hourly_rate", "2.00", "Hourly Rate")
dbutils.widgets.text("mlflow_experiment_name", "/TPC-DS Benchmark", "MLflow Experiment")
dbutils.widgets.text("run_ingestion", "true", "Run Ingestion Test")
dbutils.widgets.text("run_power_test", "true", "Run Power Test")
dbutils.widgets.text("run_concurrency", "true", "Run Concurrency Test")

In [ ]:
import os

params = dbutils.widgets.getAll()
current_path = os.getcwd()

# Fetch live warehouse configuration from Databricks REST API
warehouse_config = get_databricks_warehouse_config(
    params["dbx_warehouse_url"],
    params["dbx_pat_token"],
)
print(f"Warehouse: {warehouse_config}")

# MLflow parameters shared by all runs in this notebook
mlflow_params = {
    "scale": params["scale"],
    "warehouse_name": warehouse_config["warehouse_name"],
    "warehouse_size": warehouse_config["warehouse_size"],
    "warehouse_type": warehouse_config["warehouse_type"],
    "channel": warehouse_config.get("channel", ""),
    "num_clusters": warehouse_config.get("num_clusters", ""),
    "hourly_rate": params["dbx_hourly_rate"],
    "optimize_strategy": params["dbx_optimize_strategy"],
}

In [ ]:
%sh
set -e
WS_DIR="$PWD"
TPCDS_DIR="/tmp/tpcds"

# Prepare a local working directory (Workspace FUSE doesn't reliably persist files written by JMeter)
rm -rf "$TPCDS_DIR"
mkdir -p "$TPCDS_DIR/results" "$TPCDS_DIR/logs" "$TPCDS_DIR/test_conf"

cp "$WS_DIR"/test_conf/*.jmx "$TPCDS_DIR/test_conf/"
cp "$WS_DIR"/test_conf/*.tsv "$TPCDS_DIR/test_conf/"
cp "$WS_DIR"/test_conf/*.p8 "$TPCDS_DIR/test_conf/" 2>/dev/null || true
cp -r "$WS_DIR"/tpcds_databricks_concurrency_streams "$TPCDS_DIR/" 2>/dev/null || true

# Rewrite workspace paths in JMX files to use the local directory
sed -i "s|${WS_DIR}|${TPCDS_DIR}|g" "$TPCDS_DIR"/test_conf/*.jmx

echo "Prepared local working directory at $TPCDS_DIR"
ls -la "$TPCDS_DIR/test_conf/"

In [ ]:
if params.get("run_ingestion", "true").lower() == "true":
    run_jmeter_with_tracking(
        platform="databricks",
        test_type="ingestion",
        jmx_file="/tmp/tpcds/test_conf/databricks_ingestion.jmx",
        tsv_files=[
            "/tmp/tpcds/test_conf/tpcds-queries-ingestion-databricks.tsv",
            "/tmp/tpcds/test_conf/tpcds-queries-optimization-databricks.tsv",
        ],
        jdbc_driver="/tpcds/db_driver.jar",
        result_csvs=[
            "/tmp/tpcds/results/databricks-ingestion-report.csv",
            "/tmp/tpcds/results/databricks-optimize-report.csv",
        ],
        log_file="/tmp/tpcds/logs/jmeter-dbx-ingestion.log",
        run_params=mlflow_params,
        experiment_name=params["mlflow_experiment_name"],
        workspace_results_dir=f"{current_path}/results",
        workspace_logs_dir=f"{current_path}/logs",
    )
else:
    print("Skipping ingestion test (run_ingestion=false)")

In [ ]:
if params.get("run_power_test", "true").lower() == "true":
    run_jmeter_with_tracking(
        platform="databricks",
        test_type="power_test",
        jmx_file="/tmp/tpcds/test_conf/databricks_power_test.jmx",
        tsv_files=[
            "/tmp/tpcds/test_conf/tpcds-queries-perf-databricks.tsv",
        ],
        jdbc_driver="/tpcds/db_driver.jar",
        result_csvs=[
            "/tmp/tpcds/results/databricks-perf-test-report.csv",
        ],
        log_file="/tmp/tpcds/logs/jmeter-dbx-power-test.log",
        run_params=mlflow_params,
        experiment_name=params["mlflow_experiment_name"],
        workspace_results_dir=f"{current_path}/results",
        workspace_logs_dir=f"{current_path}/logs",
    )
else:
    print("Skipping power test (run_power_test=false)")

In [ ]:
if params.get("run_concurrency", "true").lower() == "true":
    import glob
    
    concurrency_streams = sorted(
        glob.glob("/tmp/tpcds/tpcds_databricks_concurrency_streams/stream_*.tsv")
    )
    
    run_jmeter_with_tracking(
        platform="databricks",
        test_type="concurrency",
        jmx_file="/tmp/tpcds/test_conf/databricks_concurrency.jmx",
        tsv_files=concurrency_streams,
        jdbc_driver="/tpcds/db_driver.jar",
        result_csvs=[
            "/tmp/tpcds/results/databricks-concurrency-report.csv",
        ],
        log_file="/tmp/tpcds/logs/jmeter-dbx-concurrency.log",
        run_params=mlflow_params,
        experiment_name=params["mlflow_experiment_name"],
        workspace_results_dir=f"{current_path}/results",
        workspace_logs_dir=f"{current_path}/logs",
    )
else:
    print("Skipping concurrency test (run_concurrency=false)")